# Working with Zenodo EMG Dataset (Record 14182446)

This notebook demonstrates how to:
1. Load EMG data from Zenodo record 14182446
2. Preprocess and visualize the signals
3. Extract features
4. Train and evaluate muscle fatigue detection models

## Setup

In [ ]:
# Import required libraries
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import ZenodoDataLoader
from src.preprocessing import EMGPreprocessor
from src.feature_extraction import EMGFeatureExtractor
from src.pipeline import FatigueDetectionPipeline, train_multiple_models
from src.visualization import plot_emg_signal, plot_confusion_matrix, plot_model_comparison
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Download and Load Zenodo Dataset

First, we'll load the EMG data from the Zenodo dataset.

In [ ]:
# Initialize the Zenodo data loader
loader = ZenodoDataLoader("14182446")

# Try to get record information
print("Fetching Zenodo record information...")
info = loader.get_record_info()

if info:
    print(f"\nTitle: {info.get('title', 'N/A')}")
    print(f"\nAvailable files:")
    for file_info in info.get('files', []):
        filename = file_info.get('key')
        size_mb = file_info.get('size', 0) / (1024 * 1024)
        print(f"  - {filename} ({size_mb:.2f} MB)")
else:
    print("\nCould not fetch record info automatically.")
    print("Please visit: https://zenodo.org/records/14182446")
    print("And download files to: data/zenodo/")

In [ ]:
# Load the dataset from local directory
data_dir = "../data/zenodo"

# Check if data exists
if os.path.exists(data_dir) and os.listdir(data_dir):
    print(f"Loading data from {data_dir}...")
    dataset = loader.load_dataset(data_dir)
    print(f"\nLoaded {len(dataset)} file(s)")
    
    # Display information about each file
    for filename, (signals, labels) in dataset.items():
        print(f"\n{filename}:")
        print(f"  - Signals shape: {signals.shape}")
        if labels is not None:
            print(f"  - Labels shape: {labels.shape}")
            print(f"  - Unique labels: {np.unique(labels)}")
            print(f"  - Label distribution: {np.bincount(labels.astype(int))}")
else:
    print(f"\nData directory '{data_dir}' is empty or does not exist.")
    print("\nTo proceed, please:")
    print("1. Visit https://zenodo.org/records/14182446")
    print("2. Download all files")
    print(f"3. Place them in {data_dir}")
    print("\nThen re-run this notebook.")
    dataset = None

## 2. Visualize Raw EMG Signals

Let's visualize some of the raw EMG signals from the dataset.

In [ ]:
if dataset:
    # Get the first signal
    first_file = list(dataset.keys())[0]
    signals, labels = dataset[first_file]
    
    # Plot a sample of the signal
    sample_length = min(5000, len(signals))  # Plot up to 5 seconds at 1000 Hz
    sample_signal = signals[:sample_length]
    
    plt.figure(figsize=(14, 4))
    time = np.arange(len(sample_signal)) / 1000  # Assuming 1000 Hz sampling rate
    plt.plot(time, sample_signal)
    plt.xlabel('Time (s)')
    plt.ylabel('EMG Amplitude')
    plt.title(f'Raw EMG Signal - {first_file}')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## 3. Preprocess Signals

Apply preprocessing steps: bandpass filtering, normalization, and segmentation.

In [ ]:
if dataset:
    # Initialize preprocessor (adjust sampling_rate if needed)
    sampling_rate = 1000  # Common EMG sampling rate
    preprocessor = EMGPreprocessor(sampling_rate=sampling_rate)
    
    # Process the first signal
    sample_signal = signals[:sample_length]
    
    # Apply bandpass filter
    filtered_signal = preprocessor.bandpass_filter(sample_signal)
    
    # Normalize
    normalized_signal = preprocessor.normalize(filtered_signal)
    
    # Visualize preprocessing steps
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Raw signal
    axes[0].plot(time, sample_signal)
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title('Raw EMG Signal')
    axes[0].grid(True)
    
    # Filtered signal
    axes[1].plot(time, filtered_signal)
    axes[1].set_ylabel('Amplitude')
    axes[1].set_title('Bandpass Filtered (20-450 Hz)')
    axes[1].grid(True)
    
    # Normalized signal
    axes[2].plot(time, normalized_signal)
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylabel('Amplitude')
    axes[2].set_title('Normalized Signal')
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

## 4. Feature Extraction

Extract features from the EMG signals.

In [ ]:
if dataset:
    # Initialize feature extractor
    extractor = EMGFeatureExtractor(sampling_rate=sampling_rate)
    
    # Extract features from normalized signal
    features = extractor.extract_all_features(normalized_signal)
    
    # Display features
    print("Extracted Features:")
    print("=" * 50)
    for feature_name, value in features.items():
        print(f"{feature_name:20s}: {value:.4f}")
    
    # Visualize features as a bar plot
    plt.figure(figsize=(10, 6))
    feature_names = list(features.keys())
    feature_values = list(features.values())
    
    plt.barh(feature_names, feature_values)
    plt.xlabel('Feature Value')
    plt.title('EMG Features')
    plt.tight_layout()
    plt.show()

## 5. Prepare Dataset for Training

Prepare all signals for model training.

In [ ]:
if dataset:
    # Collect all signals and labels
    signals_list = []
    labels_list = []
    
    for filename, (signals, labels) in dataset.items():
        signals_list.append(signals)
        if labels is not None:
            labels_list.append(labels)
    
    # Initialize pipeline
    pipeline = FatigueDetectionPipeline(sampling_rate=sampling_rate)
    
    # Prepare dataset (preprocess, segment, extract features)
    print("Preparing dataset...")
    features_df, labels_array = pipeline.prepare_dataset(signals_list, labels_list)
    
    print(f"\nDataset prepared:")
    print(f"  - Features shape: {features_df.shape}")
    print(f"  - Labels shape: {labels_array.shape}")
    print(f"  - Feature columns: {list(features_df.columns)}")
    print(f"  - Class distribution: {np.bincount(labels_array.astype(int))}")
    
    # Display feature statistics
    print("\nFeature Statistics:")
    print(features_df.describe())

## 6. Train and Evaluate Models

Train multiple models and compare their performance.

In [ ]:
if dataset:
    # Split data
    print("Splitting data (80% train, 20% test)...")
    X_train, X_test, y_train, y_test = train_test_split(
        features_df, labels_array,
        test_size=0.2,
        random_state=42,
        stratify=labels_array if len(np.unique(labels_array)) > 1 else None
    )
    
    print(f"  - Training set: {X_train.shape[0]} samples")
    print(f"  - Test set: {X_test.shape[0]} samples")
    
    # Train multiple models
    print("\nTraining models...")
    results = train_multiple_models(X_train, y_train, X_test, y_test)
    
    # Display results
    print("\nModel Performance:")
    print("=" * 70)
    
    results_df = pd.DataFrame({
        'Model': list(results.keys()),
        'Accuracy': [r['metrics']['accuracy'] for r in results.values()],
        'Precision': [r['metrics']['precision'] for r in results.values()],
        'Recall': [r['metrics']['recall'] for r in results.values()],
        'F1-Score': [r['metrics']['f1'] for r in results.values()]
    })
    
    print(results_df.to_string(index=False))

## 7. Visualize Results

Visualize model performance and confusion matrices.

In [ ]:
if dataset:
    # Plot model comparison
    plt.figure(figsize=(12, 6))
    
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    x = np.arange(len(results))
    width = 0.2
    
    for i, metric in enumerate(metrics):
        values = [r['metrics'][metric] for r in results.values()]
        plt.bar(x + i * width, values, width, label=metric.capitalize())
    
    plt.xlabel('Model')
    plt.ylabel('Score')
    plt.title('Model Performance Comparison')
    plt.xticks(x + width * 1.5, list(results.keys()))
    plt.legend()
    plt.ylim(0, 1.1)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Plot confusion matrices
    fig, axes = plt.subplots(1, len(results), figsize=(15, 4))
    
    for idx, (model_name, result) in enumerate(results.items()):
        cm = result['confusion_matrix']
        ax = axes[idx] if len(results) > 1 else axes
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_title(f'{model_name}\nConfusion Matrix')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    
    plt.tight_layout()
    plt.show()

## 8. Save Best Model

Save the best performing model for future use.

In [ ]:
if dataset:
    # Find best model
    best_model_name = max(results.items(), key=lambda x: x[1]['metrics']['accuracy'])[0]
    best_accuracy = results[best_model_name]['metrics']['accuracy']
    
    print(f"Best Model: {best_model_name}")
    print(f"Accuracy: {best_accuracy:.4f}")
    
    # Save the best model
    model_map = {'KNN': 'knn', 'SVM': 'svm', 'Logistic Regression': 'logistic'}
    best_model_type = model_map[best_model_name]
    
    # Train final model on all data
    final_pipeline = FatigueDetectionPipeline(sampling_rate=sampling_rate, model_type=best_model_type)
    final_pipeline.train(features_df, labels_array)
    
    # Save model
    os.makedirs('../models', exist_ok=True)
    model_path = '../models/zenodo_best_model.pkl'
    final_pipeline.save_model(model_path)
    
    print(f"\nModel saved to: {model_path}")
    print("\nYou can now use this model to make predictions on new EMG signals!")

## 9. Make Predictions

Use the trained model to make predictions on new signals.

In [ ]:
if dataset:
    # Take a sample signal for prediction
    test_signal = signals[:10000]  # 10 seconds at 1000 Hz
    
    # Make prediction
    prediction = final_pipeline.predict(test_signal)
    
    print(f"Prediction for test signal: {prediction[0]}")
    print(f"\n0 = Non-fatigued")
    print(f"1 = Fatigued")
    
    # Visualize the prediction
    plt.figure(figsize=(14, 4))
    time = np.arange(len(test_signal)) / sampling_rate
    plt.plot(time, test_signal)
    plt.xlabel('Time (s)')
    plt.ylabel('EMG Amplitude')
    plt.title(f'Test Signal - Predicted: {"Fatigued" if prediction[0] == 1 else "Non-fatigued"}')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## Summary

In this notebook, we:
1. Loaded EMG data from Zenodo record 14182446
2. Visualized raw EMG signals
3. Applied preprocessing (filtering, normalization)
4. Extracted features (time and frequency domain)
5. Trained multiple ML models (KNN, SVM, Logistic Regression)
6. Compared model performance
7. Saved the best model
8. Made predictions on new signals

For more information, see:
- [data/ZENODO_DATA.md](../data/ZENODO_DATA.md) - Detailed Zenodo dataset guide
- [examples/load_zenodo_data.py](../examples/load_zenodo_data.py) - Command-line example
- [README.md](../README.md) - Main documentation